In [19]:
import anndata
import numpy as np

In [2]:
liver = anndata.io.read_h5ad(
    '/Users/val/Library/CloudStorage/GoogleDrive-valentine.svensson@gmail.com/My Drive/Single cell data/GSE166504/GSE166504.h5ad'
)

In [3]:
liver

AnnData object with n_obs × n_vars = 82168 × 25127
    obs: 'FileName', 'CellType', 'CellID'

In [4]:
brain = anndata.io.read_h5ad(
    '/Users/val/Library/CloudStorage/GoogleDrive-valentine.svensson@gmail.com/My Drive/Single cell data/GSE212576/GSE212576.h5ad'
)

In [5]:
brain

AnnData object with n_obs × n_vars = 109826 × 32285
    obs: 'barcode', 'fname', 'batch'
    var: 'gene_ids', 'gene_names'

In [9]:
idx_ = liver.obs.sample(80_000).index
liver = liver[idx_].copy()

idx_ = brain.obs.sample(80_000).index
brain = brain[idx_].copy()

In [13]:
shared_genes = liver.var.index.intersection(brain.var.index)

In [15]:
liver = liver[:, shared_genes].copy()
brain = brain[:, shared_genes].copy()

In [23]:
# Split liver into 10k test and 70k train samples
perm = np.random.permutation(liver.n_obs)
test_idx = perm[:10_000]
train_idx = perm[10_000:]
liver_train = liver[train_idx].copy()
liver_test = liver[test_idx].copy()

# Split brain into 10k test and 70k train samples
perm_brain = np.random.permutation(brain.n_obs)
test_idx_brain = perm_brain[:10_000]
train_idx_brain = perm_brain[10_000:]
brain_train = brain[train_idx_brain].copy()
brain_test = brain[test_idx_brain].copy()

In [24]:
liver_train.write_h5ad("250508.liver_train.GSE166504.h5ad")
liver_test.write_h5ad("250508.liver_test.GSE166504.h5ad")
brain_train.write_h5ad("250508.brain_train.GSE212576.h5ad")
brain_test.write_h5ad("250508.brain_test.GSE212576.h5ad")

In [27]:
from scipy.sparse import issparse
import numpy as np

def calculate_sparsity(adata):
    X = adata.X
    total_elements = X.shape[0] * X.shape[1]
    nonzero_count = X.nnz if issparse(X) else np.count_nonzero(X)
    return (total_elements - nonzero_count) / total_elements

for adata, name in [
    (brain, 'brain'),
    (brain_test, 'brain_test'),
    (brain_train, 'brain_train'),
    (liver, 'liver'),
    (liver_test, 'liver_test'),
    (liver_train, 'liver_train')
]:
    sparsity = calculate_sparsity(adata)
    print(f"{name}: sparsity = {sparsity:.4f}")

brain: sparsity = 0.8692
brain_test: sparsity = 0.8692
brain_train: sparsity = 0.8692
liver: sparsity = 0.9408
liver_test: sparsity = 0.9410
liver_train: sparsity = 0.9408


In [25]:
ls -lh

total 7751712
-rw-r--r--@ 1 val  staff   4.7K May  8 21:41 250508 Make train test splits.ipynb
-rw-r--r--@ 1 val  staff   328M May  8 21:40 250508.brain_test.GSE212576.h5ad
-rw-r--r--@ 1 val  staff   2.2G May  8 21:40 250508.brain_train.GSE212576.h5ad
-rw-r--r--@ 1 val  staff   148M May  8 21:40 250508.liver_test.GSE166504.h5ad
-rw-r--r--@ 1 val  staff   1.0G May  8 21:40 250508.liver_train.GSE166504.h5ad
